# mem0 Drop-In Wrapper: Learn Playbooks From Your Existing mem0 Traffic

> **Time:** ~10 minutes | **Level:** Beginner

If your agent already uses [mem0](https://mem0.ai)'s managed platform, `reflexio.mem0` lets
Reflexio learn playbooks and profiles from the same traffic — **by changing one import**.

In this notebook you'll:
1. **Switch** the import and keep every mem0 call site unchanged
2. **Add** a conversation — stored in mem0 *and* published to Reflexio
3. **Search** — mem0's payload comes back untouched, plus Reflexio context as sibling keys
4. **Verify** both sides end to end, including graceful degradation

### Prerequisites
- Reflexio server running (`uv run reflexio services start --only backend`)
- `MEM0_API_KEY` set in your environment (a real mem0 platform key)
- The mem0 extra installed: `pip install 'reflexio-ai[mem0]'` (or `uv run --with mem0ai ...`)

In [1]:
import os
import time
import uuid

from _display_helpers import *

# Each run uses a unique ID so the notebook is idempotent
RUN_ID = uuid.uuid4().hex[:8]
USER_ID = f"mem0_demo_customer_{RUN_ID}"

url, reflexio_api_key = load_env()
assert os.environ.get("MEM0_API_KEY"), "Set MEM0_API_KEY before running this notebook"
show_success(f"mem0 key present, test user: {USER_ID}")

OK Environment loaded — server URL: http://127.0.0.1:18061

OK mem0 key present, test user: mem0_demo_customer_ae7f3bd9

## The One-Line Migration

```python
# Before
from mem0 import MemoryClient
# After
from reflexio.mem0 import MemoryClient
```

The wrapped client **is** a `mem0.MemoryClient` (a subclass), so every method, attribute,
and `isinstance` check behaves exactly as before. Reflexio credentials come from
`REFLEXIO_URL` / `REFLEXIO_API_KEY` env vars, or an explicit `reflexio_client=`.

In [2]:
import mem0

from reflexio import ReflexioClient
from reflexio.mem0 import MemoryClient

reflexio_client = ReflexioClient(url_endpoint=url, api_key=reflexio_api_key)
client = MemoryClient(
    api_key=os.environ["MEM0_API_KEY"],
    reflexio_client=reflexio_client,
)

assert isinstance(client, mem0.MemoryClient)
show_success("Wrapped client constructed — still a genuine mem0.MemoryClient")

OK Wrapped client constructed — still a genuine mem0.MemoryClient

## Add a Conversation

`add()` behaves exactly like mem0's: memories are extracted and stored in your mem0 account,
and the call returns mem0's own payload. In addition, the wrapper publishes the conversation
to Reflexio (best-effort, `wait_for_response=False`) so it can learn playbooks:

| mem0 argument | Reflexio field |
|---|---|
| `user_id` | `user_id` (**required** for the publish) |
| `run_id` | `session_id` |
| `agent_id` | `agent_version` |

We publish **8 turns**: Reflexio's default extraction gate (`stride_size=8`) waits for
enough new interactions before running profile/playbook extraction, so short test
conversations below that threshold won't produce profiles.


In [3]:
conversation = [
    {"role": "user", "content": "Hi, I ordered the espresso machine last week and it arrived with a cracked water tank. I travel Mondays to Wednesdays, so any replacement has to be delivered Thursday or Friday."},
    {"role": "assistant", "content": "Sorry about the damage! I've arranged a replacement with Thursday delivery and added a note that you're only available Thursdays and Fridays."},
    {"role": "user", "content": "Great. Also, please always email me the invoice as a PDF attachment — I can't open the web links from my work laptop."},
    {"role": "assistant", "content": "Done — invoices for your account will be sent as PDF attachments from now on."},
    {"role": "user", "content": "One more thing: I'm lactose intolerant, so when you suggest coffee recipes or accessories, skip anything dairy-based — oat milk works for me."},
    {"role": "assistant", "content": "Noted! I'll only recommend dairy-free options like oat milk for recipes and steaming accessories."},
    {"role": "user", "content": "And can you address me as Sam rather than my full name? Also I'm in the Pacific time zone, so please don't schedule calls before 9am PT."},
    {"role": "assistant", "content": "Of course, Sam! I've noted your Pacific time zone and will keep any calls after 9am PT."},
]

mem0_result = client.add(
    conversation,
    user_id=USER_ID,
    run_id=f"support-session-{RUN_ID}",
    agent_id="support-agent-v1",
)
show_json(mem0_result, title="mem0 add() payload (unchanged by the wrapper)")

### mem0 add() payload (unchanged by the wrapper)

{
  "event_id": "5c767cf6-f6cb-4c07-abe2-30a52a109779",
  "status": "PENDING"
}

## Verify Side 1: Memories Landed in mem0

Everything below uses plain mem0 API calls — the wrapper forwards them verbatim.
mem0 extracts memories asynchronously, so we poll briefly.

In [4]:
mem0_memories = []
for _ in range(20):
    page = client.get_all(filters={"user_id": USER_ID})
    mem0_memories = page.get("results", [])
    if mem0_memories:
        break
    time.sleep(3)

assert mem0_memories, "mem0 returned no memories for the test user"
show_json([m.get("memory") for m in mem0_memories], title=f"{len(mem0_memories)} memories in mem0")

### 3 memories in mem0

[
  "User travels Monday through Wednesday, so any replacement delivery must be scheduled for Thursday or Friday",
  "User requests that invoices always be emailed as PDF attachments because web links cannot be opened from their 
work laptop",
  "User ordered an espresso machine around July 29, 2026, and it arrived with a cracked water tank"
]

## Verify Side 2: The Trace Landed in Reflexio

The same `add()` call published the conversation to Reflexio. Query Reflexio directly:

In [5]:
stored = reflexio_client.search_interactions(user_id=USER_ID, most_recent_k=10)
assert stored.interactions, "Reflexio has no interactions for the test user"
show_interactions(stored.interactions, title=f"{len(stored.interactions)} interactions published to Reflexio")

### 8 interactions published to Reflexio

,#,Role,Content,Tools,Action
0,1,User,"Hi, I ordered the espresso machine last week a...",—,—
1,2,Assistant,Sorry about the damage! I've arranged a replac...,—,—
2,3,User,"Great. Also, please always email me the invoic...",—,—
3,4,Assistant,Done — invoices for your account will be sent ...,—,—
4,5,User,"One more thing: I'm lactose intolerant, so whe...",—,—
5,6,Assistant,Noted! I'll only recommend dairy-free options ...,—,—
6,7,User,And can you address me as Sam rather than my f...,—,—
7,8,Assistant,"Of course, Sam! I've noted your Pacific time z...",—,—


## Search: mem0 Results + Reflexio Learnings in One Call

`search()` returns mem0's payload **untouched**, plus three sibling keys:
`reflexio_profiles`, `reflexio_user_playbooks`, `reflexio_agent_playbooks`.
Code that only reads `results` is unaffected; read the new keys with `.get(...)`.

Reflexio extracts profiles/playbooks asynchronously after the publish, so we poll until
profile extraction finishes (or a timeout passes — the sibling keys are present either way).

In [6]:
deadline = time.time() + 180
search_result = None
while time.time() < deadline:
    search_result = client.search(
        "How should this customer's deliveries and invoices be handled?",
        filters={"user_id": USER_ID},
    )
    assert "reflexio_profiles" in search_result, "sibling keys missing from search payload"
    if search_result["reflexio_profiles"] or search_result["reflexio_user_playbooks"]:
        break
    time.sleep(10)

show_json([m.get("memory") for m in search_result.get("results", [])], title="mem0 search results")
show_json(
    {
        "reflexio_profiles": [p["content"] for p in search_result["reflexio_profiles"]],
        "reflexio_user_playbooks": [p["content"] for p in search_result["reflexio_user_playbooks"]],
        "reflexio_agent_playbooks": [p["content"] for p in search_result["reflexio_agent_playbooks"]],
    },
    title="Reflexio sibling keys",
)
if not (search_result["reflexio_profiles"] or search_result["reflexio_user_playbooks"]):
    show_error("Extraction still pending after timeout — sibling keys are present but empty")
else:
    show_success("Reflexio learned from the mem0 traffic and returned it alongside mem0 results")

### mem0 search results

[
  "User requests that invoices always be emailed as PDF attachments because web links cannot be opened from their 
work laptop",
  "User travels Monday through Wednesday, so any replacement delivery must be scheduled for Thursday or Friday",
  "User ordered an espresso machine around July 29, 2026, and it arrived with a cracked water tank"
]

### Reflexio sibling keys

{
  "reflexio_profiles": [
    "travels Mondays through Wednesdays, so deliveries must be scheduled for Thursday or Friday",
    "uses a work laptop that cannot open web links, so invoices must be sent as PDF email attachments",
    "prefers to be addressed as Sam"
  ],
  "reflexio_user_playbooks": [],
  "reflexio_agent_playbooks": []
}

OK Reflexio learned from the mem0 traffic and returned it alongside mem0 results

## Graceful Degradation

If Reflexio is down or misconfigured, the wrapper must never break your mem0 code path:
`add()` still stores memories, `search()` returns exactly what unwrapped mem0 returns
(no sibling keys), and failures are logged as warnings.

In [7]:
degraded = MemoryClient(
    api_key=os.environ["MEM0_API_KEY"],
    reflexio_client=ReflexioClient(url_endpoint="http://127.0.0.1:1", timeout=2),
)
r = degraded.add("I prefer morning deliveries.", user_id=USER_ID, run_id=f"degraded-{RUN_ID}")
s = degraded.search("deliveries", filters={"user_id": USER_ID})
# Managed mem0 add() returns an async event envelope; search() returns {"results": [...]}
assert isinstance(r, dict) and r
assert "results" in s
assert "reflexio_profiles" not in s
show_success("Reflexio unreachable → mem0 calls unaffected, sibling keys absent")

Best-effort Reflexio publish failed: HTTPConnectionPool(host='127.0.0.1', port=1): Max retries exceeded with url: /api/publish_interaction (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=1): Failed to establish a new connection: [Errno 61] Connection refused"))


Best-effort Reflexio search failed: HTTPConnectionPool(host='127.0.0.1', port=1): Max retries exceeded with url: /api/search (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=1): Failed to establish a new connection: [Errno 61] Connection refused"))


OK Reflexio unreachable → mem0 calls unaffected, sibling keys absent

## Cleanup

Unwrapped methods like `delete_all` delegate straight to mem0:

In [8]:
client.delete_all(user_id=USER_ID)
# mem0 deletion is asynchronous server-side; poll briefly.
remaining = client.get_all(filters={"user_id": USER_ID}).get("results", [])
for _ in range(10):
    if not remaining:
        break
    time.sleep(3)
    remaining = client.get_all(filters={"user_id": USER_ID}).get("results", [])
show_success(f"Requested deletion of test user's mem0 memories — {len(remaining)} still visible (deletion is async)")


OK Requested deletion of test user's mem0 memories — 0 still visible (deletion is async)

## What This Validated

- **Drop-in**: one import change; the wrapped client is a genuine `mem0.MemoryClient`
- **Dual write**: one `add()` stored memories in mem0 *and* published the trace to Reflexio
- **Dual read**: one `search()` returned mem0 memories plus Reflexio profiles/playbooks
- **Safety**: a dead Reflexio endpoint never breaks the mem0 path

Next: see [03_playbook.ipynb](03_playbook.ipynb) for how Reflexio aggregates and governs
the playbooks it learns from this traffic.